# 01. 데이터 확인 및 전처리

목적: 거래 데이터 로드, 검증, 전처리

- 야후 파이낸스에서 데이터 다운로드
- 데이터 품질 확인 (결측치, 이상치)
- 기본 통계 분석
- 백테스트용 데이터셋 준비

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import yfinance as yf
from datetime import datetime, timedelta

import warnings
warnings.filterwarnings('ignore')

plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 10

## 데이터 다운로드

In [ ]:
# 다운로드할 종목 (예시)
tickers = ['005930.KS', '051910.KS']  # 삼성전자, LG화학

# 데이터 기간
end_date = datetime.now()
start_date = end_date - timedelta(days=365)  # 1년

# 다운로드
data = {}
for ticker in tickers:
    print(f"다운로드 중: {ticker}...")
    df = yf.download(ticker, start=start_date, end=end_date, progress=False)
    data[ticker] = df
    print(f"  {len(df)} 거래일 로드됨")

# 첫 번째 종목 확인
first_ticker = tickers[0]
print(f"\n{first_ticker} 샘플 데이터:")
print(data[first_ticker].head())

## 데이터 품질 검증

In [ ]:
# 결측치 확인
for ticker in tickers:
    df = data[ticker]
    missing = df.isnull().sum()
    print(f"{ticker}:")
    print(f"  총 행 수: {len(df)}")
    print(f"  결측치: {missing.sum()}")
    print(f"  기간: {df.index[0]} ~ {df.index[-1]}")
    print()

## 기본 통계

In [ ]:
# 종목별 수익률 계산
for ticker in tickers:
    df = data[ticker].copy()
    
    # 수익률 (로그 수익률)
    df['returns'] = np.log(df['Close']).diff()
    
    print(f"{ticker} 통계:")
    print(f"  시작가: {df['Close'].iloc[0]:.0f}원")
    print(f"  종가: {df['Close'].iloc[-1]:.0f}원")
    print(f"  누적 수익률: {(df['Close'].iloc[-1] / df['Close'].iloc[0] - 1) * 100:.2f}%")
    print(f"  일평균 수익률: {df['returns'].mean() * 100:.4f}%")
    print(f"  일 변동성: {df['returns'].std() * 100:.2f}%")
    print()

## 가격 추이 시각화

In [ ]:
# 종목별 가격 추이
for ticker in tickers:
    df = data[ticker]
    
    plt.figure(figsize=(12, 6))
    plt.plot(df.index, df['Close'], label='Close Price')
    plt.title(f'{ticker} 가격 추이')
    plt.xlabel('Date')
    plt.ylabel('Price (KRW)')
    plt.legend()
    plt.grid(True, alpha=0.3)
    plt.tight_layout()
    plt.show()

## 데이터 저장 (백테스트용)

In [ ]:
# CSV로 저장
import os

os.makedirs('../data/processed', exist_ok=True)

for ticker in tickers:
    df = data[ticker].copy()
    df['returns'] = np.log(df['Close']).diff()
    
    filename = f'../data/processed/{ticker.split(".")[0]}_1y.csv'
    df.to_csv(filename)
    print(f"저장됨: {filename}")